# 05 — Cross-ethnic models

Does a model trained on one population work on another? Two directions are
trained and reported, and each is reported twice:

- **Within-cohort hold-out** — trained on 70% of a cohort, scored on the
  remaining 30%. This is the model's ceiling on its own population.
- **Cross-cohort** — the same model scored on the *entire* other cohort, which
  it has never seen. This is what transferability actually looks like.

Reporting both matters because the two are easy to confuse and differ
substantially: a hold-out score says how well the model has learned its own
population, and only the cross-cohort score says whether that learning carries
to another one. Each table below is labelled for which it is.

Hyperparameters come from `configs/hyperparameters.yaml`. They were found once by
Bayesian optimisation and are not re-tuned on every run; the entry point for
re-running the search is at the bottom of this notebook.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl

from src.data.io import load_hyperparameters, output_path, processed_path
from src.features.selection import drop_target_derived
from src.logging_utils import configure_logging
from src.models.evaluate import compute_metrics, score_external
from src.models.train import (
    build_classifier,
    build_voting_classifier,
    split_features_target,
    split_frames,
    stratified_split,
)

configure_logging(ROOT / "logs")

ALGORITHMS = ["CatBoost", "XGBoost", "lightGBM"]
HYPERPARAMETERS = load_hyperparameters()["cross_ethnic"]

cohorts = {
    "NHANES": drop_target_derived(
        pl.read_parquet(processed_path("NHANES_race1_features.parquet"))
    ),
    "KNHANES": drop_target_derived(
        pl.read_parquet(processed_path("KNHANES_race2_features.parquet"))
    ),
}
for name, frame in cohorts.items():
    print(f"{name}: {frame.shape}, IR+ {frame['IR'].mean():.4f}")

NHANES: (11660, 242), IR+ 0.4445
KNHANES: (15138, 242), IR+ 0.2792


## Within-cohort hold-out

Each cohort is split 70/30 with stratification, and three boosted-tree models
plus a soft-voting ensemble are trained on the training portion. The voting
ensemble refits its members, so it depends only on their hyperparameters and the
training data.

In [2]:
results = []
trained = {}

for cohort, frame in cohorts.items():
    params = HYPERPARAMETERS[cohort.lower()]
    X, y = split_features_target(frame)
    X_train, X_test, y_train, y_test = stratified_split(X, y)

    fitted = []
    for algorithm in ALGORITHMS:
        model = build_classifier(algorithm, y_train, params[algorithm])
        model.fit(X_train, y_train)
        fitted.append((algorithm, model))

    ensemble = build_voting_classifier(
        [(f"model_{index}", model) for index, (_, model) in enumerate(fitted)]
    )
    ensemble.fit(X_train, y_train)
    fitted.append(("Voting", ensemble))

    runs = {}
    for algorithm, model in fitted:
        preds = model.predict_proba(X_test)[:, 1]
        metrics = compute_metrics(y_test, preds)
        runs[algorithm] = {
            "model": model,
            "preds": preds,
            "metrics": metrics,
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
        }
        results.append(
            {
                "evaluation": "within-cohort hold-out",
                "trained_on": cohort,
                "evaluated_on": f"{cohort} (30% hold-out)",
                "model": algorithm,
                "n_samples": len(y_test),
                **{k: v for k, v in metrics.items() if k not in ("confusion_matrix", "optimal_threshold")},
                **metrics["confusion_matrix"],
            }
        )
    trained[cohort] = runs
    print(f"{cohort}: trained on {len(X_train):,} rows, scored on {len(X_test):,}")

NHANES: trained on 8,162 rows, scored on 3,498


KNHANES: trained on 10,596 rows, scored on 4,542


## Cross-cohort

The best model from each direction is scored on the **whole** of the other
cohort — CatBoost for the NHANES-trained direction, XGBoost for the
KNHANES-trained one.

In [3]:
# Every model is scored on the cohort it never saw, in both directions. Reporting
# one algorithm per direction would confound the direction of transfer with the
# choice of algorithm, which is the comparison this table exists to make.
for cohort in cohorts:
    other = "KNHANES" if cohort == "NHANES" else "NHANES"
    for algorithm in trained[cohort]:
        metrics = score_external(trained[cohort][algorithm]["model"], cohorts[other])
        results.append(
            {
                "evaluation": "cross-cohort",
                "trained_on": cohort,
                "evaluated_on": f"{other} (all)",
                "model": algorithm,
                "n_samples": metrics["n_samples"],
                **{
                    k: v
                    for k, v in metrics.items()
                    if k not in ("confusion_matrix", "optimal_threshold", "preds", "n_samples")
                },
                **metrics["confusion_matrix"],
            }
        )

cross_ethnic = pl.DataFrame(results)
cross_ethnic.to_pandas().to_excel(output_path("cross_ethnic.xlsx"), index=False)
cross_ethnic

2026-09-22 22:24:13 [INFO] src.models.evaluate: External scoring on 15138 samples: AUC=0.862


2026-09-22 22:24:13 [INFO] src.models.evaluate: External scoring on 15138 samples: AUC=0.861


2026-09-22 22:24:14 [INFO] src.models.evaluate: External scoring on 15138 samples: AUC=0.861


2026-09-22 22:24:14 [INFO] src.models.evaluate: External scoring on 15138 samples: AUC=0.862


2026-09-22 22:24:14 [INFO] src.models.evaluate: External scoring on 11660 samples: AUC=0.861


2026-09-22 22:24:14 [INFO] src.models.evaluate: External scoring on 11660 samples: AUC=0.860


2026-09-22 22:24:14 [INFO] src.models.evaluate: External scoring on 11660 samples: AUC=0.861


2026-09-22 22:24:14 [INFO] src.models.evaluate: External scoring on 11660 samples: AUC=0.862


evaluation,trained_on,evaluated_on,model,n_samples,roc_auc,accuracy,sensitivity(recall),specificity,PPV(precision),NPV,f1_score,Youdens_Index,pr_auc,TP,TN,FP,FN
str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64
"""within-cohort hold-out""","""NHANES""","""NHANES (30% hold-out)""","""CatBoost""",3498,0.868,0.788,0.76,0.81,0.762,0.808,0.761,0.57,0.85,1182,1573,370,373
"""within-cohort hold-out""","""NHANES""","""NHANES (30% hold-out)""","""XGBoost""",3498,0.867,0.787,0.758,0.81,0.761,0.807,0.76,0.568,0.848,1179,1573,370,376
"""within-cohort hold-out""","""NHANES""","""NHANES (30% hold-out)""","""lightGBM""",3498,0.867,0.79,0.759,0.814,0.766,0.808,0.762,0.573,0.851,1180,1582,361,375
"""within-cohort hold-out""","""NHANES""","""NHANES (30% hold-out)""","""Voting""",3498,0.868,0.79,0.758,0.815,0.766,0.808,0.762,0.573,0.851,1178,1584,359,377
"""within-cohort hold-out""","""KNHANES""","""KNHANES (30% hold-out)""","""CatBoost""",4542,0.878,0.798,0.781,0.805,0.608,0.905,0.683,0.586,0.767,990,2635,639,278
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""cross-cohort""","""NHANES""","""KNHANES (all)""","""Voting""",15138,0.862,0.812,0.626,0.884,0.676,0.859,0.65,0.51,0.733,2646,9643,1269,1580
"""cross-cohort""","""KNHANES""","""NHANES (all)""","""CatBoost""",11660,0.861,0.735,0.895,0.606,0.645,0.878,0.75,0.501,0.834,4638,3927,2550,545
"""cross-cohort""","""KNHANES""","""NHANES (all)""","""XGBoost""",11660,0.86,0.736,0.89,0.613,0.648,0.875,0.75,0.503,0.832,4614,3971,2506,569


Transfer costs little in AUC — 0.868 → 0.862 one way, 0.878 → 0.860 the other —
but the operating point moves sharply. Scoring KNHANES with the NHANES model
trades sensitivity for specificity (0.760 → 0.617, 0.810 → 0.892); the reverse
direction does the opposite (0.783 → 0.890, 0.807 → 0.613). The threshold of 0.5
is calibrated to the training cohort's prevalence, which differs: 44% against
28%.

## Re-running the hyperparameter search

Off by default. The published models were tuned once and their parameters
committed; this exists so the search can be repeated, not so it runs every time.
Set `RETUNE = True` to tune one combination and compare the result against the
committed values.

In [4]:
RETUNE = False

if RETUNE:
    from src.models.train import tune_hyperparameters

    cohort, algorithm = "NHANES", "XGBoost"
    X, y = split_features_target(cohorts[cohort])
    X_train, _, y_train, _ = stratified_split(X, y)

    recovered = tune_hyperparameters(X_train, y_train, algorithm)
    committed = HYPERPARAMETERS[cohort.lower()][algorithm]
    for key in sorted(set(recovered) | set(committed)):
        mark = "==" if recovered.get(key) == committed.get(key) else "!="
        print(f"{key:20s} {recovered.get(key)!r} {mark} {committed.get(key)!r}")